### Tuning for automatic parameter optimization
In this notebook, we will explore the tuning functionalities in Qewton. 
Since in this workshop multiple people work on the same GPUs, we use a simple toy example for this demonstration.

The problem is given by the ODE:
\begin{align*}
\partial_t u &= 4\pi \cos(4\pi t) + 2t \quad \text{in } (0,2), \\
u(0) &= 0 
\end{align*}

In [ ]:
# This block is for GPU selection. Please execute.
# import pathlib
# import os
# user = int(str(pathlib.Path().resolve())[22:24])
# os.environ["CUDA_VISIBLE_DEVICES"]= str(user % 4)

In [ ]:
import qewton
import math 

# Variables like always
T = qewton.Variable("t", dim=1)
U = qewton.Variable("u", dim=1)

# The interval also
interval = qewton.geometries.Interval(T, 0, 2)

Now we want to automatically determine how many points to use for the inner sampler in our PINN pipeline.
We can achieve this by using a discrete hyperparameter for the number of points.

In the tuning process, multiple neural networks will be trained with different numbers of points for the inner sampler, allowing us to find the optimal configuration.

In [ ]:
n_points_param = qewton.DiscreteHyperparameter((100, 5000), name="Inner Sampler Points")
inner_sampler = qewton.RandomUniformSampler(interval, n_points=n_points_param)

left_sampler = qewton.GridSampler(interval.boundary_left, 1)

Next we also want to vary the layer and neuron number in the neural network.

In [ ]:
# TODO: Define two hyperparameters for the neural network: hidden neurons and hidden layers
hidden_neurons_param = 
n_hidden_layers_param = 

model = qewton.FCN(
    in_neurons=T,
    hidden_neurons=hidden_neurons_param,
    out_neurons=U,
    n_hidden_layers=n_hidden_layers_param
)

For training we need the same conditions as before:

In [ ]:
cos = qewton.bb.Cos()

# TODO: Define the residual function for the ODE
def residual_fun(u: U, t: T):
    return

constraint = qewton.PINNConstraint(residual_fun, name="PINNConstraint")
ode_graph = qewton.PINNPipeline(inner_sampler, [model], constraint)

In [ ]:
# The initial condition is already finished
def boundary_residual_fun(u: U):
    return u

# TODO: define a ContinuousHyperparameter for the boundary weight
boundary_weight_param = 

boundary_constraint = qewton.constraints.PINNConstraint(
    boundary_residual_fun, 
    name="InitialConstraint", 
    weight=boundary_weight_param
)
boundary_graph = qewton.PINNPipeline(left_sampler, [model], boundary_constraint)

To compare different hyperparameter configurations, we need some condition that evaluates the learned solution. We could just check which configuration minimizes the residual of the ODE over the training domain.

For this example however, we will compare with the real analytical solution. *Note*, we train physics informed, but validate with the real solution!

In [ ]:
validation_constraint = qewton.MSEConstraint(name="ValidationConstraint", 
                                             evaluated_in_mode=qewton.EvaluationPhase.VALIDATION)
validation_pipeline = qewton.Graph()

# Real solution
Sin = qewton.bb.Sin()
def real_solution(t: T):
    return Sin(4.0*math.pi*t) + t**2

with validation_pipeline.tracker(0):
    t = inner_sampler()
    model_out = model(t)
    validation_constraint(model_out, real_solution(t))

Now we define a trainer, but not yet start the training process. We will first set up the trainer with the defined pipelines and constraints.

In [ ]:
adam_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.Adam(),
    lr=0.001,
    max_iterations=1000,
)

trainer = qewton.optim.GraphBasedTrainer(
    optimization_phases=adam_phase,
    graphs=[boundary_graph, ode_graph, validation_pipeline],
    training_objectives=[boundary_constraint, constraint],
    callbacks=[
        qewton.optim.ModelParameterSizeCallback([model])
    ],
    device=qewton.cuda(0),
)

Now the trainer is passed to a *Tuner* instance, which will handle the hyperparameter optimization process. For this multiple trainings will be started with different hyperparameter configurations to find the best one.

**Best one** means which one has the smallest validation error compared to the real solution.

In [ ]:
save_path = "results"   

tuner = qewton.optim.tuner.GridSearchTuner(
    trainer,
    tuning_objectives=[validation_constraint],
    trial_number=40, # Number of hyperparameter configurations to try
    devices=[
        "cuda:0",
        "cuda:1",
        "cuda:2",
        "cuda:3",
    ],
    save_path=save_path,
    trials_per_device=1,
)
tuner.run()

Now we can analyze the results of the tuning:

In [ ]:
tuning_analyzer = qewton.optim.TuningAnalyzer(save_path)
tuning_analyzer.best_run()

We can check how important the different parameters are in general to learn a good solution.
In the next cell a correlation between the hyperparameters and the validation error will be computed.
A value close to 1 equals a strong positive correlation, meaning a *bigger* value of the hyperparameter generally leads to a *smaller* validation error.

In [ ]:
tuning_analyzer.param_importance()

We can also visualize the correlation between the hyperparameters and the validation error to better understand their impact.

In [ ]:
fig = tuning_analyzer.parallel_coordinates_plot(
    axes=["Hidden Neurons", "Hidden Layer", "Inner Sampler Points", "Boundary Weight", "Parameter size [MB]"],
    objectives=["ValidationConstraint"],
    log_axes=["ValidationConstraint"]
    )
app = qewton.viz.DashApplication.create(fig)
app.run(port=8060, jupyter_mode="external")